# Job Application AI Agent using LangChain

#### AI Agent = LLM + Harness

## LangChain
 - is a Open source framework which provides the harness for LLMs.
 - Langchain is preferred when you know the sequence/chain of events 
 - Langchain agents are built on top of LangGraph

## LangGraph
 - is a low level orchestration framework for building agents or complex workflows
 - Langgraph is based on Graphs, where nodes are steps, edges are transitions and state is shared data 

## LangChain vs LangGraph

![title](comparison.png)


## Lets start building:

 * [Create Agent](#create_agent)
 * [Define Tools](#define_tools)
 * [Memory to Agents](#memory_in_agents)
 * [Define Agent State](#agent_state)
 * [Add Human in the loop - Middleware in Agents](#hitl)
 * [Multimodal Agents](#multimodal_agents)
 * [Invoking Agents](#invoke_agents)

### Load env file with API keys

In [5]:
# OPENAI_API_KEY='xxxxxxxxxxxxxxxxxxxxxx'
# Optional for evaluation and tracing
# LANGSMITH_API_KEY='xxxxxxxxxxxxxxxxxxxxxxxxxx'
# uncomment to set tracing to true when you set up your LangSmith account
# LANGSMITH_TRACING=true
# LANGSMITH_PROJECT=project_name
# Uncomment the following if you are on the EU instance:
# LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com

from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-5-nano"

### Define Tools
<a id='define_tools'></a>

In [6]:
import os
import requests
from bs4 import BeautifulSoup
from langchain.tools import tool
from job_scraper import scrape_jobs_to_csv
from linkedin_scraper import scrape_linkedin_jobs_to_csv
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.tools import tool, ToolRuntime

@tool
async def job_finder(platform: str, role: str, city: str, radius: int, runtime: ToolRuntime) -> str:
    """Find jobs based on platform the user wants to search jobs on and create a csv file with list of jobs to apply to"""
    
    csv_filename = "job_details.csv"
    csv_path = os.path.join(os.getcwd(), csv_filename)
    
    if platform.lower() == "stepstone":
        
        scrape_jobs_to_csv(
            f"https://www.stepstone.de/work/{role.strip()}/in-{city.lower()}?radius={radius}",
            csv_filename,
            max_jobs=3
        )
    else:
        await scrape_linkedin_jobs_to_csv(role, city, csv_path)
        
    return Command(update={
            "pathToJobsCsv": csv_path,
            "messages": [ToolMessage("Successfully updated csv file name. Next step is to display all the jobs", 
                                     tool_call_id=runtime.tool_call_id)]}
        )

In [7]:
@tool
def display_jobs(runtime: ToolRuntime) -> str:
    """ Display the title, company, and location of every job found so far """
    try:
        csv_path = runtime.state["pathToJobsCsv"]
    except KeyError:
        return "No CSV file path key found"    
        
    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    columns = ["title", "company", "location"]
    widths = {col: max(len(col), *(len(row[col]) for row in rows)) for col in columns}

    def format_row(values):
        return " | ".join(str(v).ljust(widths[col]) for col, v in zip(columns, values))

    lines = [format_row(columns), "-+-".join("-" * widths[col] for col in columns)]
    lines.extend(format_row([row[col] for col in columns]) for row in rows)

    return "\n".join(lines)

In [8]:
import asyncio
import csv
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

@tool
async def analyze_resume_and_make_suggestions(runtime: ToolRuntime) -> str:
    """Fetch job description and add changes to resume(passed as pdf) by tailoring it to each job description and save it to csv for human approval"""
    try:
        csv_path = runtime.state["pathToJobsCsv"]
        pdf_base64 = runtime.state["pdfBase64"]
    except KeyError:
        return "No CSV file path key found"   
        
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = reader.fieldnames + ["resume_corrections"] + ["apply"]

    llm = init_chat_model(MODEL)

    async def get_suggestions(row):
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "You are a resume coach. Compare the attached resume to the job descriptions "
                        "and suggest specific, concise corrections or additions to better match the job. "
                        "Match their language, mirror their priorities, and keep it to one page. "
                        "Don’t make anything up — only use what’s already in my resume, just position it better."
                        "Return only a short bulleted list of suggestions.\n\n"
                        f"Job Description:\n{row['description']}\n"
                    ),
                },
                {
                    "type": "file",
                    "mime_type": "application/pdf",
                    "base64": pdf_base64,
                },
            ]
        )
        response = await llm.ainvoke([message])
        row["resume_corrections"] = response.content.replace("\n", " | ")
        row["apply"] = True
            
    await asyncio.gather(*(get_suggestions(row) for row in rows))

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    
    return ToolMessage(f"""Tailored resume suggestions for all jobs and saved to csv {csv_path}. 
                    Next step: Start applying to jobs""", tool_call_id=runtime.tool_call_id)

In [9]:
@tool
def apply_jobs(runtime: ToolRuntime) -> str:
    """ Apply to jobs listed in csv post confirmation on all the jobs from human"""
    return ToolMessage("Applied to all the jobs", tool_call_id=runtime.tool_call_id)

### Define Agent State
<a id='agent_state'></a>

In [10]:
from langchain.agents import AgentState

class JobApplicationState(AgentState):
    pathToJobsCsv: str
    pdfBase64: str

In [11]:
JOB_APPLICATION_SYSTEM_PROMPT = """You are a job application assistant that helps the user find relevant jobs, \
tailor their resume, and apply to roles on their behalf.

## Tools and when to use them

- `job_finder`: search for jobs and create a csv with job details. Take the platform from users input, 
if job platform not provided default to linkedin
- `display_jobs`: use this to display jobs to the user after job search is done
- `analyze_resume_and_make_suggestions`: fetch job descriptions and use that to propose resume changes. This requires \
explicit human approval before changes are considered final — never claim a resume has been updated \
until the user has approved it.
- `apply_jobs`: Start applying to all the jobs 

## Autonomy

You have full authority to run the entire workflow — find jobs, display them, tailor the resume, apply — \
end to end without stopping to ask the user's permission or preference between steps. Use the defaults in \
the Rules section below to resolve anything ambiguous (platform, number of postings, scope of search) \
instead of asking. Do not ask questions like "should I broaden the search," "should I include LinkedIn \
too," or "do you want me to proceed" — just proceed.

The ONLY point where you must stop and wait for the user is the human-in-the-loop interrupt that fires \
automatically after `resume_updater` runs, for approving the tailored resume changes. That pause is \
enforced by the system, not something you decide to add — do not create additional approval checkpoints \
of your own.

The one exception where asking is unavoidable: if `resume_updater` is missing a required input it has no \
way to obtain on its own (e.g. no resume PDF has been provided at all), ask for exactly that one missing \
input, directly and in a single question. Do not bundle it with unrelated preference questions.

## Rules

1. Never fabricate job listings, company names, salaries, or job details. Only report what a tool actually \
returned. If a tool returns no results or fails, say so plainly and offer to retry or try a different \
platform — do not invent plausible-sounding listings to fill the gap.
2. Always use the tool that matches the platform the user names. Do not substitute one platform's tool for \
another's results.
3. When a job search and resume tool succeeds, display the csv data on jobs.
4. If asked to find jobs without a platform specified, default to LinkedIn.
5. If user does not specify the number of job postings, default to 3.
6. Complete one step at a time, in sequence: find jobs first, wait for that tool's result, then display \
jobs, then update the resume, then apply. Do not call `resume_updater` in the same turn as any other \
tool. `resume_updater` pauses the conversation for human approval, and calling it alongside another \
tool call can leave that other call's result unrecorded, which breaks the next turn. Proceed through \
these steps automatically per the Autonomy section above — sequencing them one at a time is about \
avoiding orphaned tool calls, not about pausing for permission at each one.
"""

### Middleware in Agents
<a id='hitl'></a>

 - Middleware provides a way to more tightly control what happens inside the agent
 - Middleware is useful for the following:
      - Tracking agent behavior with logging, analytics, and debugging.
      - Transforming prompts, tool selection, and output formatting.
      - Adding retries, fallbacks, and early termination logic.
      - Applying rate limits, guardrails, and PII detection.

### Memory in Agents
<a id='memory_in_agents'></a>

 - Remembers previous interactions, learns from feedback and adapt to user preferences
 - Short term memory lets your application remember previous interactions within a single thread or conversation.
 - To add short-term memory (thread-level persistence) to an agent, you need to specify a checkpointer when creating an agent.
 - LangChain’s agent manages short-term memory as a part of your agent’s state.
 - Short-term memory updates when the agent is invoked or a step (like a tool call) is completed, and the state is read at the start of each step.

### Create Agent
<a id='create_agent'></a>

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.agents.middleware import HumanInTheLoopMiddleware

job_application_agent = create_agent(
        MODEL,
        system_prompt=JOB_APPLICATION_SYSTEM_PROMPT,
        tools=[job_finder, display_jobs, analyze_resume_and_make_suggestions, apply_jobs],
        state_schema=JobApplicationState,
        checkpointer=InMemorySaver(),
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "apply_jobs": True
                }
            ),
        ],
    )


### Multimodal Agents
<a id='multimodal_agents'></a>

In [13]:
# File uploader
import base64
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.pdf', multiple=False)
display(uploader)

FileUpload(value=(), accept='.pdf', description='Upload')

In [14]:
# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
pdf_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
pdf_base64 = base64.b64encode(pdf_bytes).decode("utf-8")

### Invoke Agent

<a id='invoke_agents'></a>

In [15]:
config = {"configurable": {"thread_id": "1"}}

response = await job_application_agent.ainvoke(
    {"messages": [HumanMessage(content="""Find jobs for AI engineer in Berlin on Stepstone, 
                                        update my resume to the job description and apply to jobs""")
                 ],
     "pdfBase64": pdf_base64,
    },
    config=config
)

[1/3] scraped: AI Engineer
[2/3] scraped: AI Engineer Financial Services (m/w/d)
[3/3] scraped: AI Requirements Engineer Public Sector (m/w/d)


/Users/nirjharijankar/projects/AI/langchain/job_application_agent/.venv/lib/python3.12/site-packages/langchain_core/messages/block_translators/openai.py:109: UserWarning: OpenAI may require a filename for file uploads. Specify a filename in the content block, e.g.: {'type': 'file', 'mime_type': '...', 'base64': '...', 'filename': 'my-file.pdf'}. Using placeholder filename 'LC_AUTOGENERATED'.
  warnings.warn(


In [16]:

# Human In The Loop Interrupt Handler
if response.get("__interrupt__"):
    print(response["__interrupt__"]) 
    response = await job_application_agent.ainvoke(
        Command( 
            resume={"decisions": [{"type": "approve"}]}
        ), 
        config=config # Same thread ID to resume the paused conversation
    )

print(response['messages'][-1].content)

[Interrupt(value={'action_requests': [{'name': 'apply_jobs', 'args': {}, 'description': 'Tool execution requires approval\n\nTool: apply_jobs\nArgs: {}'}], 'review_configs': [{'action_name': 'apply_jobs', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='b97aa331da572cc9aa9e9600dc8bc3d3')]
Here’s the status of your job search and initial application flow:

- Platform: Stepstone
- Role: AI Engineer
- Location: Berlin
- Radius: 25 km
- Number of postings found: 3

Jobs found:
1) AI Engineer — commercetools GmbH — Berlin
2) AI Engineer Financial Services (m/w/d) — Sopra Steria — bundesweit
3) AI Requirements Engineer Public Sector (m/w/d) — Sopra Steria — bundesweit

Resume updates:
- I generated tailored resume suggestions based on the job descriptions and saved them to the csv at:
  /Users/nirjharijankar/projects/AI/langchain/job_application_agent/notebooks/job_details.csv
- Note: These are proposals and require your approval before any changes are applied to the act